# Harfi: Qwen3-1.7B v1 on spelling v2.0 and round 6 (Colab)

**Qwen3-1.7B** with exactly Llama v5's recipe (LoRA rank 32, alpha 64, lr 2e-4, 2 epochs, the last epoch's model kept)
on the same `harfi-v6` export (144,986 examples, spelling v2.0 + round 6) that Qwen3-0.6B v2 and Llama 3.2 1B v5 trained
on. Only the base model differs, so the three compare fairly: does a larger Qwen close Qwen 0.6B's gap to Llama on text
by other people (others test, −0.017; docs/EVAL.md, "Qwen v2 against v4")?

Qwen3 trains in fp16 on the T4. Its thinking mode is turned off in the prompt, as for the other Qwen runs. Licence: Apache-2.0.

**Output:** `arabizi-qwen3-1.7b-v1-Q4_K_M.gguf` (about 1.1 GB, the level every shipped model uses) and `-Q5_K_M.gguf`
(about 1.26 GB, kept only if it fits the 1.5 GB memory budget and scores clearly better), their `.sha256` files and
`run.json` (with the exact prompt the engine needs). No Q8_0: at about 1.8 GB it cannot ship, and for Qwen 0.6B it was
at most a fraction of a point above Q4.

**Before you start**, in Google Drive, folder `harfi` (My Drive -> harfi): `arabizi-train-v6.zip`, the training data
(the files of `build/train-export/` at the top of the zip). About 3 GB of Drive space is needed for checkpoints and the
GGUF files.

**How to run:**
1. Runtime -> Change runtime type -> **T4 GPU**.
2. Runtime -> **Run all**, and allow access to Google Drive when asked. Keep the tab open; about 8 hours of GPU in all.
3. **If Colab disconnects or stops** (idle tab, the 12-hour limit, or the free GPU quota): reconnect, or come back when
   a GPU is offered again, and **Run all** again. Training goes on from the last checkpoint in Drive (one every 269
   steps, about 15 minutes), never from the start, and ends with the same weights as one uninterrupted run. The training
   cell prints a `CHECKPOINT saved to Drive` line (time, step, folder) for each one, so you can see what a disconnect
   would keep; a resumed run first lists the checkpoints already in Drive.
4. At the end, `harfi/out/<run>/` in Drive holds the two GGUF files, their `.sha256` files and `run.json`. Download those
   to be scored.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
from google.colab import drive

drive.mount("/content/drive")
HARFI = "/content/drive/MyDrive/harfi"

In [ ]:
REF = "feat/qwen3-1.7b-v1"  # this branch until it is merged (it has checkpoints and resuming), then "main"
# Fresh copy of the code on every run (an old clone would otherwise be kept and run).
!rm -rf /content/repo
!git clone --quiet --depth 1 --branch "$REF" https://github.com/Mustafa-Abbara/arabizi-keyboard /content/repo
!git -C /content/repo log -1 --format='repo commit: %h %s'
!pip install --quiet -r /content/repo/train/requirements.txt

In [ ]:
import glob
import json
import os
import subprocess
import sys
import zipfile

REPO = "/content/repo"
DATA = "/content/data"
if not os.path.exists(f"{DATA}/export.json"):
    zipfile.ZipFile(f"{HARFI}/arabizi-train-v6.zip").extractall(DATA)
with open(f"{DATA}/export.json") as f:
    summary = json.load(f)
print(summary["rows_total"], "examples | AI share", summary["ai_share"], "| prompt v", summary["prompt_version"])
assert summary["rows_total"] == 144986, "this should be the harfi-v6 export (spelling v2.0 + round 6)"

# Checkpoints live in Drive, so a disconnect loses at most the steps since the last one.
OUT = f"{HARFI}/out"
BASE, TAG, NAME = "qwen3-1.7b", "v1", "arabizi-qwen3-1.7b-v1"
QUANTS = ["Q4_K_M", "Q5_K_M"]
os.makedirs(OUT, exist_ok=True)
# A checkpoint cut off while it was being written (no trainer_state.json, which the trainer writes last) cannot be
# resumed from: remove it, so the previous complete one is used.
import shutil
for c in glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}/checkpoints/checkpoint-*"):
    if not os.path.exists(f"{c}/trainer_state.json"):
        print("removing an unfinished checkpoint:", c)
        shutil.rmtree(c)
# The run to go on from: the latest one with a checkpoint or a finished training (a run cut off before its first
# checkpoint starts over).
runs = [r for r in sorted(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
        if glob.glob(f"{r}/checkpoints/checkpoint-*") or os.path.exists(f"{r}/run.json")]
RUN = runs[-1] if runs else None
print("going on from", RUN if RUN else "nothing: a new run")
if RUN:
    print("checkpoints already in Drive:", sorted(os.path.basename(c) for c in glob.glob(f"{RUN}/checkpoints/checkpoint-*")))

In [ ]:
# Training's output, streamed into this cell, with a line for every checkpoint that reaches Drive.
import math
import time
from datetime import datetime

train_rows = sum(f["rows"] for name, f in summary["files"].items() if name.startswith("train-"))
STEPS = math.ceil(math.ceil(train_rows / 8) / 4) * 2  # batch 8, gradient accumulation 4, 2 epochs


def saved_checkpoints():
    # A checkpoint counts once its trainer_state.json is written (the trainer writes it last).
    return {os.path.dirname(p) for p in glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}/checkpoints/checkpoint-*/trainer_state.json")}


def show_new_checkpoints(seen):
    for path in sorted(saved_checkpoints() - seen, key=lambda p: int(p.rsplit("-", 1)[-1])):
        step = int(path.rsplit("-", 1)[-1])
        print(f"[{datetime.now():%H:%M}] CHECKPOINT saved to Drive: step {step} of about {STEPS} "
              f"({step / STEPS:.0%}) -> {path}", flush=True)
        seen.add(path)


def run_streamed(cmd, watch_checkpoints=False):
    # Progress-bar lines are shown at most every 5 minutes; every other line (losses, validation, messages) as it comes.
    seen = saved_checkpoints()
    last_bar = 0.0
    env = {**os.environ, "PYTHONUNBUFFERED": "1"}
    with subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env) as p:
        for line in p.stdout:
            line = line.rstrip()
            is_bar = "it/s]" in line or "s/it]" in line
            if line and (not is_bar or time.time() - last_bar >= 300):
                print(line, flush=True)
                last_bar = time.time() if is_bar else last_bar
            if watch_checkpoints:
                show_new_checkpoints(seen)
    if watch_checkpoints:
        show_new_checkpoints(seen)
    if p.returncode:
        raise subprocess.CalledProcessError(p.returncode, cmd)

In [ ]:
# Llama v5's settings: rank 32, alpha 64, lr 2e-4, 2 epochs; a checkpoint every 269 steps (1/16 of an epoch). How often
# it saves does not change the weights. The merged model goes to the Colab disk, not Drive (3.4 GB, deleted after the
# export).
def done(run):
    return run is not None and all(os.path.exists(f"{run}/{NAME}-{q}.gguf") for q in QUANTS)


if done(RUN):
    print("already finished and exported")
else:
    if RUN and os.path.exists(f"{RUN}/run.json") and not os.path.isdir("/content/merged"):
        # Training finished but the session ended before the export: train again from the last checkpoint (at most
        # 269 steps) to get the merged model back.
        os.remove(f"{RUN}/run.json")
    if not (RUN and os.path.exists(f"{RUN}/run.json")):
        print(f"=== {BASE} {TAG}" + (", resumed" if RUN else ""), flush=True)
        # A failed run raises and stops the notebook instead of exporting nothing.
        run_streamed([sys.executable, f"{REPO}/train/finetune_lora.py", "--base", BASE, "--data", DATA, "--out", OUT,
                      "--tag", TAG, "--rank", "32", "--alpha", "64", "--epochs", "2", "--save-steps", "269",
                      "--merged-dir", "/content/merged", *(["--resume", RUN] if RUN else [])], watch_checkpoints=True)
        RUN = max(glob.glob(f"{OUT}/*-{BASE}-lora-{TAG}"))
    # Export on the Colab disk (the f16 file is 3.4 GB), then copy only the GGUF and SHA-256 files to Drive.
    os.makedirs("/content/export", exist_ok=True)
    run_streamed(["bash", f"{REPO}/train/export_gguf.sh", "/content/merged", NAME, "/content/export", " ".join(QUANTS)])
    for q in QUANTS:
        for suffix in (".gguf", ".gguf.sha256"):
            subprocess.run(["cp", f"/content/export/{NAME}-{q}{suffix}", RUN], check=True)
    assert done(RUN), "the GGUF files did not reach Drive"
    subprocess.run(["rm", "-rf", "/content/merged", "/content/export", f"{RUN}/checkpoints"], check=True)

## Results

The best validation loss is only a first look (Qwen v2's and Llama v5's are in their run.json files; same data, but
different tokenizers, so compare accuracy, not loss). Accuracy on the dev sets and the real chats decides, scored against
Llama v5 and Qwen v2 with every answer in spelling v2.0. Download the GGUF files, their `.sha256` files and `run.json` from
`harfi/out/<run>/` in Drive.

In [ ]:
with open(f"{RUN}/run.json") as f:
    r = json.load(f)
print(f"{r['run_id']}: best validation loss {r['best_eval_loss']:.4f} ({r['minutes']} min, {r['precision']})")
print([h for h in r["history"] if "eval_loss" in h])
for s in r["public_samples"]:
    print(f"  {s['arabizi']}\n    -> {s['output']}\n    want {s['reference']}")
!ls -l "$RUN"